In [ ]:
from google.colab import files
import pandas as pd
import io

uploaded = files.upload()
filename = "allframe_update_addEpige.txt"

data = pd.read_csv(
    io.BytesIO(uploaded[filename]),
    sep="\t",
    dtype=str
)

print("Rows and columns:", data.shape)

Saving allframe_update_addEpige.txt to allframe_update_addEpige.txt
Rows and columns: (245848, 53)


In [ ]:
paper = data[
    data["PMID"].str.strip().eq("28459459") # check if this row is from cameron study
    & data["Species"].str.strip().eq("Homo sapiens") # make sure cell genome is human
    & data["Cas9_type"].str.strip().eq("SpCas9") # making sure we are using this specific cas9 enzyme
].copy()

print("Records from Cameron et al.:", len(paper))

columns = [
    "On_target_site",
    "Guide_sequence",
    "Target_sequence",
    "Identity",
    "Targeted_location",
    "Targeted_method",
    "Indel_control%",
    "Indel_treatment%",
    "Indel_accu%",
    "Validation",
    "Remark4"
]

display(paper[columns].head(10))

Records from Cameron et al.: 18955


,On_target_site,Guide_sequence,Target_sequence,Identity,Targeted_location,Targeted_method,Indel_control%,Indel_treatment%,Indel_accu%,Validation,Remark4
7445,AAVS1_site14,GGGGCCACTAGGGACAGGATTGG,GGGGCCACTAGGGACAGGATTGG,ON,In cell,"AmpliSeq, sensitivity is 0.1%",NaN,NaN,85.586,TRUE,"Indel_accu% in there is from treated-control, ..."
7446,AAVS1_site14,GGGGCCACTAGGGACAGGATTGG,TGGGCCACTATGGACAGGAATGG,OFF,In cell,"AmpliSeq, sensitivity is 0.1%",NaN,NaN,NaN,NaN,"Indel_accu% in there is from treated-control, ..."
7447,AAVS1_site14,GGGGCCACTAGGGACAGGATTGG,GGGGTCACTGGGGACAAGATTGG,OFF,In cell,"AmpliSeq, sensitivity is 0.1%",NaN,NaN,1.466,TRUE,"Indel_accu% in there is from treated-control, ..."
7448,AAVS1_site14,GGGGCCACTAGGGACAGGATTGG,GGGGCCAGTAGGGAGAGGATAGG,OFF,In cell,"AmpliSeq, sensitivity is 0.1%",NaN,NaN,1.25,TRUE,"Indel_accu% in there is from treated-control, ..."
7449,AAVS1_site14,GGGGCCACTAGGGACAGGATTGG,GAGGCCACTGGGGACAGGCGTGG,OFF,In cell,"AmpliSeq, sensitivity is 0.1%",NaN,NaN,NaN,NaN,"Indel_accu% in there is from treated-control, ..."
7450,AAVS1_site14,GGGGCCACTAGGGACAGGATTGG,GGGGCCACTAGGGAAAGAGTGGG,OFF,In cell,"AmpliSeq, sensitivity is 0.1%",NaN,NaN,NaN,NaN,"Indel_accu% in there is from treated-control, ..."
7451,AAVS1_site14,GGGGCCACTAGGGACAGGATTGG,GGGGCCACTAGAGAAGGGATGGG,OFF,In cell,"AmpliSeq, sensitivity is 0.1%",NaN,NaN,NaN,NaN,"Indel_accu% in there is from treated-control, ..."
7452,AAVS1_site14,GGGGCCACTAGGGACAGGATTGG,GGGGCCAATTAGGACAGGATGGG,OFF,In cell,"AmpliSeq, sensitivity is 0.1%",NaN,NaN,27.927,TRUE,"Indel_accu% in there is from treated-control, ..."
7453,AAVS1_site14,GGGGCCACTAGGGACAGGATTGG,GGGACCACTGGGGACAGGCCTGG,OFF,In cell,"AmpliSeq, sensitivity is 0.1%",NaN,NaN,NaN,NaN,"Indel_accu% in there is from treated-control, ..."
7454,AAVS1_site14,GGGGCCACTAGGGACAGGATTGG,GGGACCACTGGGGACAGGCCTGG,OFF,In cell,"AmpliSeq, sensitivity is 0.1%",NaN,NaN,NaN,NaN,"Indel_accu% in there is from treated-control, ..."


In [ ]:
for column in [
    "Source",
    "Delivery",
    "Time",
    "Targeted_source",
    "Targeted_location",
    "Targeted_delivery",
    "Targeted_Time",
    "Validation"
]:
    print(f"\n{column}")
    print(paper[column].value_counts(dropna=False)) # Look at this column in the paper table. Count how many rows contain each different answer, and print those counts

print("\nNotes explaining the measurements:")
for note in paper["Remark4"].dropna().unique(): # removes missing or duplicates
    print(note)


Source
Source
HEK293 cells    18955
Name: count, dtype: int64

Delivery
Delivery
sgRNP Incubation    18955
Name: count, dtype: int64

Time
Time
4h    18955
Name: count, dtype: int64

Targeted_source
Targeted_source
HEK293 cells    18681
K562 cells        274
Name: count, dtype: int64

Targeted_location
Targeted_location
In cell    18955
Name: count, dtype: int64

Targeted_delivery
Targeted_delivery
Lentivirus tranduction     8847
sgRNA transfection         6868
sgRNP Transfection         2692
Linear DNA transfection     548
Name: count, dtype: int64

Targeted_Time
Targeted_Time
3d     9560
28d    3240
6d     2692
14d    2692
20d     771
Name: count, dtype: int64

Validation
Validation
NaN      15341
FALSE     3277
TRUE       337
Name: count, dtype: int64

Notes explaining the measurements:
Indel_accu% in there is from treated-control, validated off-targets are treated-control>0.1% & (control: >=500 total reads & <0.2%)


In [ ]:
paper.to_csv("Cameron_2017_extracted.csv", index=False)
files.download("Cameron_2017_extracted.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
#Each record links a guide sequence to a genomic candidate site and experimental information.
#Some records include cellular editing measurements. Repeated pairs may represent different conditions.
#This extraction has not yet been filtered or assigned final training labels.

In [ ]:
group = paper[
    paper["Targeted_source"].eq("HEK293 cells") # makes sure all four of these params match
    & paper["Targeted_delivery"].eq("sgRNA transfection")
    & paper["Targeted_Time"].eq("3d")
    & paper["Identity"].eq("OFF")
].copy()

group["editing_above_control"] = pd.to_numeric(
    group["Indel_accu%"], #if no number leave blank
    errors="coerce"
)

print("Records in selected group:", len(group))
print("\nReported outcomes:")
print(group["Validation"].value_counts(dropna=False))

display(group[[
    "Guide_sequence",
    "Target_sequence",
    "editing_above_control",
    "Validation"
]].head(10))

Records in selected group: 6859

Reported outcomes:
Validation
NaN      5171
FALSE    1577
TRUE      111
Name: count, dtype: int64


,Guide_sequence,Target_sequence,editing_above_control,Validation
7446,GGGGCCACTAGGGACAGGATTGG,TGGGCCACTATGGACAGGAATGG,NaN,NaN
7447,GGGGCCACTAGGGACAGGATTGG,GGGGTCACTGGGGACAAGATTGG,1.466,TRUE
7448,GGGGCCACTAGGGACAGGATTGG,GGGGCCAGTAGGGAGAGGATAGG,1.250,TRUE
7449,GGGGCCACTAGGGACAGGATTGG,GAGGCCACTGGGGACAGGCGTGG,NaN,NaN
7450,GGGGCCACTAGGGACAGGATTGG,GGGGCCACTAGGGAAAGAGTGGG,NaN,NaN
7451,GGGGCCACTAGGGACAGGATTGG,GGGGCCACTAGAGAAGGGATGGG,NaN,NaN
7452,GGGGCCACTAGGGACAGGATTGG,GGGGCCAATTAGGACAGGATGGG,27.927,TRUE
7453,GGGGCCACTAGGGACAGGATTGG,GGGACCACTGGGGACAGGCCTGG,NaN,NaN
7454,GGGGCCACTAGGGACAGGATTGG,GGGACCACTGGGGACAGGCCTGG,NaN,NaN
7455,GGGGCCACTAGGGACAGGATTGG,GGGACCACTGGGCACAGGATCGG,0.139,TRUE


In [ ]:
# Keep rows with a numeric editing measurement
# and a reported TRUE/FALSE outcome.
reported_answer = (
    group["Validation"]
    .str.strip()
    .str.upper()
)

measured = group[
    group["editing_above_control"].notna()
    & reported_answer.isin(["TRUE", "FALSE"])
].copy()

print("Rows in your experimental group:", len(group))
print("Rows with measurements and answers:", len(measured))

print("\nReported answers:")
print(
    measured["Validation"]
    .str.strip()
    .str.upper()
    .value_counts()
)

Rows in your experimental group: 6859
Rows with measurements and answers: 1688

Reported answers:
Validation
FALSE    1577
TRUE      111
Name: count, dtype: int64


In [ ]:
# Standardize the sequence letters.
for column in ["Guide_sequence", "Target_sequence"]:
    measured[column] = measured[column].str.strip().str.upper()

# Require 23 DNA letters: 20 targeting bases + 3 PAM bases.
valid_sequences = (
    measured["Guide_sequence"].str.fullmatch(r"[ACGT]{23}", na=False)
    & measured["Target_sequence"].str.fullmatch(r"[ACGT]{23}", na=False)
)

# Exclude records annotated as having alignment bulges.
no_bulges = (
    measured["Bulge"].fillna("").str.strip().eq("")
    & measured["Bulge2"].fillna("").str.strip().eq("")
)

sequence_checked = measured[
    valid_sequences & no_bulges
].copy()

print("Before sequence checks:", len(measured))
print("After sequence checks:", len(sequence_checked))
print(
    sequence_checked["Validation"]
    .str.strip()
    .str.upper()
    .value_counts()
)

Before sequence checks: 1688
After sequence checks: 1685
Validation
FALSE    1577
TRUE      108
Name: count, dtype: int64


In [ ]:
sequence_checked["reported_label"] = (
    sequence_checked["Validation"]
    .str.strip()
    .str.upper()
)

pair_summary = (
    sequence_checked
    .groupby(["Guide_sequence", "Target_sequence"])
    .agg(
        records=("reported_label", "size"),
        different_answers=("reported_label", "nunique")
    )
    .reset_index()
)

print("Distinct guide–DNA pairs:", len(pair_summary))
print("Pairs appearing more than once:",
      (pair_summary["records"] > 1).sum())
print("Pairs with conflicting answers:",
      (pair_summary["different_answers"] > 1).sum())

Distinct guide–DNA pairs: 1664
Pairs appearing more than once: 15
Pairs with conflicting answers: 1


In [ ]:
repeated_pairs = pair_summary[
    pair_summary["records"] > 1
]

repeated_records = sequence_checked.merge(
    repeated_pairs,
    on=["Guide_sequence", "Target_sequence"],
    how="inner"
)

columns = [
    "Guide_sequence",
    "Target_sequence",
    "Target_region",
    "On_target_site",
    "editing_above_control",
    "reported_label",
    "different_answers"
]

# Show the conflicting pair first.
display(
    repeated_records[columns]
    .sort_values(
        ["different_answers", "Guide_sequence", "Target_sequence"],
        ascending=[False, True, True]
    )
)

,Guide_sequence,Target_sequence,Target_region,On_target_site,editing_above_control,reported_label,different_answers
21,GCATACAGTGATTTGATGAAAGG,GGACACAGAGATTTGATGAAGGG,chr1:146983733-146983755,PAPSS2,0.172,TRUE,2
22,GCATACAGTGATTTGATGAAAGG,GGACACAGAGATTTGATGAAGGG,chr1:149486935-149486957,PAPSS2,0.117,TRUE,2
23,GCATACAGTGATTTGATGAAAGG,GGACACAGAGATTTGATGAAGGG,chr1:16568409-16568431,PAPSS2,0.089,FALSE,2
24,GCATACAGTGATTTGATGAAAGG,GGACACAGAGATTTGATGAAGGG,chr1:144429036-144429058,PAPSS2,0.000,FALSE,2
25,GCATACAGTGATTTGATGAAAGG,GGACACAGAGATTTGATGAAGGG,chr1:145580026-145580048,PAPSS2,0.080,FALSE,2
26,GCATACAGTGATTTGATGAAAGG,GGACACAGAGATTTGATGAAGGG,chr1:146127455-146127477,PAPSS2,0.121,TRUE,2
28,GCATACAGTGATTTGATGAAAGG,GGACACAGAGATTTGATGAAGGG,chr1:145394314-145394336,PAPSS2,0.266,TRUE,2
6,CTCGTCTGATAAGACAACAGTGG,ACTTTCTGTGAAGACAACTGGGG,chr20:29073523-29073545,PTPRC_site2,-0.009,FALSE,1
7,CTCGTCTGATAAGACAACAGTGG,ACTTTCTGTGAAGACAACTGGGG,chr22:12595587-12595609,PTPRC_site2,-0.008,FALSE,1
8,CTCGTCTGATAAGACAACAGTGG,GAAGTCTGATGAGACAACAGAGC,chr10:111186674-111186696,PTPRC_site2,-0.005,FALSE,1


In [ ]:
site_keys = [
    "Guide_sequence",
    "Target_sequence",
    "Target_region"
]

same_site_records = sequence_checked[
    sequence_checked.duplicated(site_keys, keep=False)
]

print("Records sharing the same guide, DNA, and location:",
      len(same_site_records))

display(same_site_records[[
    *site_keys,
    "editing_above_control",
    "reported_label",
    "Remark1",
    "Remark3"
]])

Records sharing the same guide, DNA, and location: 2


,Guide_sequence,Target_sequence,Target_region,editing_above_control,reported_label,Remark1,Remark3
7633,GGGGCCACTAGGGACAGGATTGG,GTGGCCAGCTGGGACAGGATAGG,chr9:34350223-34350245,0.032,FALSE,"for this gene, pick 68 petential off-targets t...",64 nM
7634,GGGGCCACTAGGGACAGGATTGG,GTGGCCAGCTGGGACAGGATAGG,chr9:34350223-34350245,0.032,FALSE,"for this gene, pick 68 petential off-targets t...",64 nM


In [ ]:
# Compare all columns in the two records.
differences = same_site_records.columns[
    same_site_records.nunique(dropna=False) > 1
].tolist()

print("Columns that differ:", differences)

if not differences:
    working_data = sequence_checked.drop_duplicates().copy()
    print("Exact duplicate removed.")
    print("Remaining records:", len(working_data))
else:
    display(same_site_records[differences])

Columns that differ: []
Exact duplicate removed.
Remaining records: 1684


In [ ]:
from google.colab import files

working_data.to_csv(
    "Cameron_2017_working.csv",
    index=False
)

files.download("Cameron_2017_working.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
verified_data = working_data[
    working_data["On_target_site"] != "AAVS1_site14" #this one is dropped bc you can't confirm these records against the workbook
].copy()

print("Verified records:", len(verified_data))
print(verified_data["reported_label"].value_counts())

Verified records: 1622
reported_label
FALSE    1541
TRUE       81
Name: count, dtype: int64


In [ ]:
# The answer the model will learn to predict
verified_data["label"] = verified_data["reported_label"].map({
    "TRUE": 1,
    "FALSE": 0
})

# Separate the sequence from its final three letters
verified_data["guide_20"] = verified_data["Guide_sequence"].str[:20]
verified_data["target_20"] = verified_data["Target_sequence"].str[:20]
verified_data["target_pam"] = verified_data["Target_sequence"].str[20:]

display(
    verified_data[
        ["guide_20", "target_20", "target_pam", "label"]
    ].head()
)

print("Missing labels:", verified_data["label"].isna().sum())
print("Different guides:", verified_data["guide_20"].nunique())

,guide_20,target_20,target_pam,label
12072,GGGTGGGGGGAGTTTGCTCC,GTGGGGGTAGAGTTTGCTCC,AGG,1
12073,GGGTGGGGGGAGTTTGCTCC,AGGGAGGAGGAATTTGCTCC,AGG,1
12074,GGGTGGGGGGAGTTTGCTCC,TAGTGGAGGGAGCTTGCTCC,TGG,1
12075,GGGTGGGGGGAGTTTGCTCC,GGATGGAGGGAGTTTGCTCC,TGG,1
12077,GGGTGGGGGGAGTTTGCTCC,AAGTAAGGGAAGTTTGCTCC,TGG,1


Missing labels: 0
Different guides: 8


In [ ]:
verified_data["mismatch_count"] = [
    sum(g != t for g, t in zip(guide, target))
    for guide, target in zip(
        verified_data["guide_20"],
        verified_data["target_20"]
    )
]

display(
    verified_data[
        ["guide_20", "target_20", "mismatch_count", "label"]
    ].head()
)

print("Examples by mismatch count:")
print(
    verified_data["mismatch_count"]
    .value_counts()
    .sort_index()
)

,guide_20,target_20,mismatch_count,label
12072,GGGTGGGGGGAGTTTGCTCC,GTGGGGGTAGAGTTTGCTCC,4,1
12073,GGGTGGGGGGAGTTTGCTCC,AGGGAGGAGGAATTTGCTCC,5,1
12074,GGGTGGGGGGAGTTTGCTCC,TAGTGGAGGGAGCTTGCTCC,4,1
12075,GGGTGGGGGGAGTTTGCTCC,GGATGGAGGGAGTTTGCTCC,2,1
12077,GGGTGGGGGGAGTTTGCTCC,AAGTAAGGGAAGTTTGCTCC,5,1


Examples by mismatch count:
mismatch_count
2      20
3      94
4     249
5     303
6     253
7     163
8     157
9     140
10    113
11     74
12     38
13     16
14      2
Name: count, dtype: int64


In [ ]:
# One column per position:
# 1 = different letters, 0 = matching letters
position_columns = []

for position in range(20):
    column = f"mismatch_pos_{position + 1}"
    position_columns.append(column)

    verified_data[column] = (
        verified_data["guide_20"].str[position]
        != verified_data["target_20"].str[position]
    ).astype(int)

display(
    verified_data[
        ["mismatch_count"] + position_columns
    ].head()
)

# These position indicators should add up to the mismatch count
print(
    "Position counts match:",
    (
        verified_data[position_columns].sum(axis=1)
        == verified_data["mismatch_count"]
    ).all()
)

,mismatch_count,mismatch_pos_1,mismatch_pos_2,mismatch_pos_3,mismatch_pos_4,mismatch_pos_5,mismatch_pos_6,mismatch_pos_7,mismatch_pos_8,mismatch_pos_9,...,mismatch_pos_11,mismatch_pos_12,mismatch_pos_13,mismatch_pos_14,mismatch_pos_15,mismatch_pos_16,mismatch_pos_17,mismatch_pos_18,mismatch_pos_19,mismatch_pos_20
12072,4,0,1,0,1,0,0,0,1,1,...,0,0,0,0,0,0,0,0,0,0
12073,5,1,0,0,1,1,0,0,1,0,...,0,1,0,0,0,0,0,0,0,0
12074,4,1,1,0,0,0,0,1,0,0,...,0,0,1,0,0,0,0,0,0,0
12075,2,0,0,1,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
12077,5,1,1,0,0,1,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0


Position counts match: True


In [ ]:
guide_summary = (
    verified_data.groupby("guide_20")
    .agg(
        total_examples=("label", "size"),
        positive_examples=("label", "sum")
    )
)

guide_summary["negative_examples"] = (
    guide_summary["total_examples"]
    - guide_summary["positive_examples"]
)

display(guide_summary)

,total_examples,positive_examples,negative_examples
guide_20,,,
ATAGGAGAAGATGATGTATA,178,11,167
CTCGTCTGATAAGACAACAG,156,1,155
GCAAAACTCAACCCTACCCC,204,2,202
GCATACAGTGATTTGATGAA,182,9,173
GCTGATGTAGTCACTCTTGA,127,0,127
GGAATCCCTTCTGCAGCACC,175,7,168
GGGTGGGGGGAGTTTGCTCC,422,42,380
GGTGGACAAGCGGCAGATAG,178,9,169


In [ ]:
validation_guides = {"GCATACAGTGATTTGATGAA"}  # PAPSS2
test_guides = {
    "ATAGGAGAAGATGATGTATA",  # CD34
    "GGAATCCCTTCTGCAGCACC"   # FANCF
}

validation_data = verified_data[
    verified_data["guide_20"].isin(validation_guides)
].copy()

test_data = verified_data[
    verified_data["guide_20"].isin(test_guides)
].copy()

train_data = verified_data[
    ~verified_data["guide_20"].isin(
        validation_guides | test_guides
    )
].copy()

# Check that no guide appears in two groups
train_guides = set(train_data["guide_20"])
assert train_guides.isdisjoint(validation_guides)
assert train_guides.isdisjoint(test_guides)
assert validation_guides.isdisjoint(test_guides)

for name, data in [
    ("Training", train_data),
    ("Validation", validation_data),
    ("Test", test_data)
]:
    print(
        f"{name}: {len(data)} records, "
        f"{data['label'].sum()} positives, "
        f"{data['guide_20'].nunique()} guides"
    )

print("No guides overlap.")

Training: 1087 records, 54 positives, 5 guides
Validation: 182 records, 9 positives, 1 guides
Test: 353 records, 18 positives, 2 guides
No guides overlap.


In [ ]:
# first model starts learning
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score

feature_columns = ["mismatch_count"] + position_columns

baseline_model = make_pipeline(
    StandardScaler(),
    LogisticRegression(
        class_weight="balanced",
        max_iter=2000,
        random_state=42
    )
)

# Learn using only the training group
baseline_model.fit(
    train_data[feature_columns],
    train_data["label"]
)

# Evaluate on the validation group
validation_scores = baseline_model.predict_proba(
    validation_data[feature_columns]
)[:, 1]

print("Baseline model trained.")
print(
    "Validation average precision:",
    round(
        average_precision_score(
            validation_data["label"],
            validation_scores
        ),
        3
    )
)
print(
    "Positive fraction for comparison:",
    round(validation_data["label"].mean(), 3)
)

Baseline model trained.
Validation average precision: 0.115
Positive fraction for comparison: 0.049


In [ ]:
# Count each of the 12 possible mismatch types
mismatch_type_columns = []

for guide_base in "ACGT":
    for target_base in "ACGT":
        if guide_base == target_base:
            continue

        column = f"mismatch_{guide_base}_to_{target_base}"
        mismatch_type_columns.append(column)

        verified_data[column] = [
            sum(
                g == guide_base and t == target_base
                for g, t in zip(guide, target)
            )
            for guide, target in zip(
                verified_data["guide_20"],
                verified_data["target_20"]
            )
        ]

print(
    "Mismatch type counts match:",
    (
        verified_data[mismatch_type_columns].sum(axis=1)
        == verified_data["mismatch_count"]
    ).all()
)

display(
    verified_data[
        ["mismatch_count"] + mismatch_type_columns
    ].head()
)

Mismatch type counts match: True


,mismatch_count,mismatch_A_to_C,mismatch_A_to_G,mismatch_A_to_T,mismatch_C_to_A,mismatch_C_to_G,mismatch_C_to_T,mismatch_G_to_A,mismatch_G_to_C,mismatch_G_to_T,mismatch_T_to_A,mismatch_T_to_C,mismatch_T_to_G
12072,4,0,0,0,0,0,0,1,0,2,0,0,1
12073,5,0,0,0,0,0,0,4,0,0,0,0,1
12074,4,0,0,0,0,0,0,2,0,1,0,1,0
12075,2,0,0,0,0,0,0,2,0,0,0,0,0
12077,5,0,0,0,0,0,0,5,0,0,0,0,0


In [ ]:
# Refresh the same fixed splits with the new features
validation_data = verified_data[
    verified_data["guide_20"].isin(validation_guides)
].copy()

test_data = verified_data[
    verified_data["guide_20"].isin(test_guides)
].copy()

train_data = verified_data[
    ~verified_data["guide_20"].isin(
        validation_guides | test_guides
    )
].copy()

feature_columns = (
    ["mismatch_count"]
    + position_columns
    + mismatch_type_columns
)

# Keep the earlier result for comparison
previous_ap = average_precision_score(
    validation_data["label"],
    validation_scores
)

baseline_model = make_pipeline(
    StandardScaler(),
    LogisticRegression(
        class_weight="balanced",
        max_iter=2000,
        random_state=42
    )
)

baseline_model.fit(
    train_data[feature_columns],
    train_data["label"]
)

validation_scores = baseline_model.predict_proba(
    validation_data[feature_columns]
)[:, 1]

updated_ap = average_precision_score(
    validation_data["label"],
    validation_scores
)

print("Earlier validation average precision:", round(previous_ap, 3))
print("With mismatch types:", round(updated_ap, 3))
print("Number of input features:", len(feature_columns))

Earlier validation average precision: 0.115
With mismatch types: 0.123
Number of input features: 33


In [ ]:
import numpy as np

base_to_index = {"A": 0, "C": 1, "G": 2, "T": 3}

def encode_sequence_pairs(data):
    # 8 channels: 4 for the guide and 4 for the target
    encoded = np.zeros((len(data), 8, 20), dtype=np.float32)

    for row_number, (guide, target) in enumerate(
        zip(data["guide_20"], data["target_20"])
    ):
        for position, (g, t) in enumerate(zip(guide, target)):
            encoded[row_number, base_to_index[g], position] = 1
            encoded[row_number, 4 + base_to_index[t], position] = 1

    return encoded

X_train_sequence = encode_sequence_pairs(train_data)
X_validation_sequence = encode_sequence_pairs(validation_data)
X_test_sequence = encode_sequence_pairs(test_data)

print("Training shape:", X_train_sequence.shape)
print("Validation shape:", X_validation_sequence.shape)
print("Test shape:", X_test_sequence.shape)

assert np.all(X_train_sequence[:, :4, :].sum(axis=1) == 1)
assert np.all(X_train_sequence[:, 4:, :].sum(axis=1) == 1)
print("Training encoding checks passed.")

Training shape: (1087, 8, 20)
Validation shape: (182, 8, 20)
Test shape: (353, 8, 20)
Training encoding checks passed.


In [ ]:
import torch
from torch import nn

torch.manual_seed(42)

class SequenceCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.pattern_layers = nn.Sequential(
            nn.Conv1d(8, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv1d(16, 16, kernel_size=3, padding=1),
            nn.ReLU()
        )

        self.output_layers = nn.Sequential(
            nn.Flatten(),
            nn.Linear(16 * 20, 16),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(16, 1)
        )

    def forward(self, sequences):
        patterns = self.pattern_layers(sequences)
        return self.output_layers(patterns).squeeze(1)

cnn_model = SequenceCNN()

# Check that one input example produces one output score
cnn_model.eval()
with torch.no_grad():
    sample_output = cnn_model(
        torch.from_numpy(X_train_sequence[:5])
    )

print("Model created.")
print("Output shape:", sample_output.shape)
print(
    "Trainable parameters:",
    sum(p.numel() for p in cnn_model.parameters())
)

Model created.
Output shape: torch.Size([5])
Trainable parameters: 6337


In [ ]:
import copy
from torch.utils.data import TensorDataset, DataLoader

# Make the training run repeatable
torch.manual_seed(42)
cnn_model = SequenceCNN()

train_sequences = torch.from_numpy(X_train_sequence)
train_labels = torch.tensor(
    train_data["label"].to_numpy(),
    dtype=torch.float32
)

validation_sequences = torch.from_numpy(X_validation_sequence)

train_loader = DataLoader(
    TensorDataset(train_sequences, train_labels),
    batch_size=64,
    shuffle=True
)

# Give rare positive examples more weight
positive_count = train_labels.sum().item()
negative_count = len(train_labels) - positive_count

loss_function = nn.BCEWithLogitsLoss(
    pos_weight=torch.tensor(negative_count / positive_count)
)

optimizer = torch.optim.Adam(
    cnn_model.parameters(),
    lr=0.001,
    weight_decay=0.001
)

best_ap = -1
best_weights = None
best_epoch = None

for epoch in range(1, 31):
    cnn_model.train()
    total_loss = 0

    for sequences, labels in train_loader:
        optimizer.zero_grad()
        raw_scores = cnn_model(sequences)
        loss = loss_function(raw_scores, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * len(labels)

    cnn_model.eval()
    with torch.no_grad():
        scores = torch.sigmoid(
            cnn_model(validation_sequences)
        ).numpy()

    validation_ap = average_precision_score(
        validation_data["label"], scores
    )

    if validation_ap > best_ap:
        best_ap = validation_ap
        best_epoch = epoch
        best_weights = copy.deepcopy(cnn_model.state_dict())

    if epoch == 1 or epoch % 5 == 0:
        print(
            f"Round {epoch}: "
            f"training loss = {total_loss / len(train_labels):.3f}, "
            f"validation AP = {validation_ap:.3f}"
        )

cnn_model.load_state_dict(best_weights)
cnn_model.eval()

print("\nBest round:", best_epoch)
print("Best CNN validation AP:", round(best_ap, 3))
print("Logistic regression validation AP:", round(updated_ap, 3))

Round 1: training loss = 1.305, validation AP = 0.067
Round 5: training loss = 1.093, validation AP = 0.090
Round 10: training loss = 0.998, validation AP = 0.105
Round 15: training loss = 0.938, validation AP = 0.111
Round 20: training loss = 0.911, validation AP = 0.127
Round 25: training loss = 0.864, validation AP = 0.128
Round 30: training loss = 0.763, validation AP = 0.118

Best round: 25
Best CNN validation AP: 0.128
Logistic regression validation AP: 0.123


In [ ]:
class CombinedModel(nn.Module):
    def __init__(self, number_of_features):
        super().__init__()

        self.sequence_branch = nn.Sequential(
            nn.Conv1d(8, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv1d(16, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Flatten(),
            nn.Linear(16 * 20, 16),
            nn.ReLU()
        )

        self.feature_branch = nn.Sequential(
            nn.Linear(number_of_features, 8),
            nn.ReLU()
        )

        self.prediction_head = nn.Sequential(
            nn.Dropout(0.2),
            nn.Linear(16 + 8, 1)
        )

    def forward(self, sequences, features):
        sequence_output = self.sequence_branch(sequences)
        feature_output = self.feature_branch(features)

        combined = torch.cat(
            [sequence_output, feature_output],
            dim=1
        )

        return self.prediction_head(combined).squeeze(1)

# Learn scaling from training data only
feature_scaler = StandardScaler()

X_train_features = feature_scaler.fit_transform(
    train_data[feature_columns]
).astype(np.float32)

X_validation_features = feature_scaler.transform(
    validation_data[feature_columns]
).astype(np.float32)

torch.manual_seed(42)
combined_model = CombinedModel(len(feature_columns))

combined_model.eval()
with torch.no_grad():
    sample_output = combined_model(
        torch.from_numpy(X_train_sequence[:5]),
        torch.from_numpy(X_train_features[:5])
    )

print("Combined model created.")
print("Output shape:", sample_output.shape)

Combined model created.
Output shape: torch.Size([5])


In [ ]:
torch.manual_seed(42)
combined_model = CombinedModel(len(feature_columns))

combined_loader = DataLoader(
    TensorDataset(
        torch.from_numpy(X_train_sequence),
        torch.from_numpy(X_train_features),
        train_labels
    ),
    batch_size=64,
    shuffle=True
)

combined_optimizer = torch.optim.Adam(
    combined_model.parameters(),
    lr=0.001,
    weight_decay=0.001
)

validation_features = torch.from_numpy(X_validation_features)

combined_best_ap = -1
combined_best_weights = None
combined_best_epoch = None

for epoch in range(1, 31):
    combined_model.train()
    total_loss = 0

    for sequences, features, labels in combined_loader:
        combined_optimizer.zero_grad()

        raw_scores = combined_model(sequences, features)
        loss = loss_function(raw_scores, labels)

        loss.backward()
        combined_optimizer.step()

        total_loss += loss.item() * len(labels)

    combined_model.eval()
    with torch.no_grad():
        scores = torch.sigmoid(
            combined_model(
                validation_sequences,
                validation_features
            )
        ).numpy()

    validation_ap = average_precision_score(
        validation_data["label"], scores
    )

    if validation_ap > combined_best_ap:
        combined_best_ap = validation_ap
        combined_best_epoch = epoch
        combined_best_weights = copy.deepcopy(
            combined_model.state_dict()
        )

    if epoch == 1 or epoch % 5 == 0:
        print(
            f"Round {epoch}: "
            f"training loss = {total_loss / len(train_labels):.3f}, "
            f"validation AP = {validation_ap:.3f}"
        )

# Restore the best version
combined_model.load_state_dict(combined_best_weights)
combined_model.eval()

print("\nBest combined-model round:", combined_best_epoch)
print("Logistic regression AP:", round(updated_ap, 3))
print("CNN-only AP:", round(best_ap, 3))
print("Combined-model AP:", round(combined_best_ap, 3))

Round 1: training loss = 1.311, validation AP = 0.189
Round 5: training loss = 1.118, validation AP = 0.096
Round 10: training loss = 1.013, validation AP = 0.105
Round 15: training loss = 0.934, validation AP = 0.134
Round 20: training loss = 0.895, validation AP = 0.136
Round 25: training loss = 0.840, validation AP = 0.153
Round 30: training loss = 0.732, validation AP = 0.166

Best combined-model round: 2
Logistic regression AP: 0.123
CNN-only AP: 0.128
Combined-model AP: 0.192


In [ ]:
from sklearn.metrics import precision_recall_curve

cnn_model.eval()
combined_model.eval()

with torch.no_grad():
    cnn_validation_scores = torch.sigmoid(
        cnn_model(validation_sequences)
    ).numpy()

    combined_validation_scores = torch.sigmoid(
        combined_model(
            validation_sequences,
            validation_features
        )
    ).numpy()

baseline_validation_scores = baseline_model.predict_proba(
    validation_data[feature_columns]
)[:, 1]

def choose_threshold(labels, scores):
    precision, recall, thresholds = precision_recall_curve(
        labels, scores
    )

    # The last precision/recall point has no threshold
    f1 = (
        2 * precision[:-1] * recall[:-1]
        / (precision[:-1] + recall[:-1] + 1e-12)
    )

    best_index = np.argmax(f1)
    return float(thresholds[best_index])

validation_predictions = {
    "Logistic regression": baseline_validation_scores,
    "CNN only": cnn_validation_scores,
    "Combined model": combined_validation_scores
}

model_thresholds = {
    name: choose_threshold(validation_data["label"], scores)
    for name, scores in validation_predictions.items()
}

for name, threshold in model_thresholds.items():
    print(f"{name}: threshold = {threshold:.3f}")

Logistic regression: threshold = 0.544
CNN only: threshold = 0.336
Combined model: threshold = 0.513


In [ ]:
import pandas as pd
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    roc_auc_score
)

# Apply the scaling learned from training data
X_test_features = feature_scaler.transform(
    test_data[feature_columns]
).astype(np.float32)

test_sequences = torch.from_numpy(X_test_sequence)
test_features = torch.from_numpy(X_test_features)
test_labels = test_data["label"].to_numpy()

cnn_model.eval()
combined_model.eval()

with torch.no_grad():
    cnn_test_scores = torch.sigmoid(
        cnn_model(test_sequences)
    ).numpy()

    combined_test_scores = torch.sigmoid(
        combined_model(test_sequences, test_features)
    ).numpy()

baseline_test_scores = baseline_model.predict_proba(
    test_data[feature_columns]
)[:, 1]

test_scores = {
    "Logistic regression": baseline_test_scores,
    "CNN only": cnn_test_scores,
    "Combined model": combined_test_scores
}

results = []

for name, scores in test_scores.items():
    predictions = (
        scores >= model_thresholds[name]
    ).astype(int)

    results.append({
        "Model": name,
        "Precision": precision_score(
            test_labels, predictions, zero_division=0
        ),
        "Recall": recall_score(
            test_labels, predictions, zero_division=0
        ),
        "F1": f1_score(
            test_labels, predictions, zero_division=0
        ),
        "Average precision": average_precision_score(
            test_labels, scores
        ),
        "ROC-AUC": roc_auc_score(test_labels, scores)
    })

test_results = pd.DataFrame(results).set_index("Model")
display(test_results.round(3))

print("Test examples:", len(test_labels))
print("Test positives:", int(test_labels.sum()))

,Precision,Recall,F1,Average precision,ROC-AUC
Model,,,,,
Logistic regression,0.115,0.333,0.171,0.114,0.635
CNN only,0.000,0.000,0.000,0.046,0.461
Combined model,0.000,0.000,0.000,0.065,0.562


Test examples: 353
Test positives: 18


In [ ]:
from sklearn.metrics import confusion_matrix

counts = []

for name, scores in test_scores.items():
    predictions = (
        scores >= model_thresholds[name]
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        test_labels, predictions, labels=[0, 1]
    ).ravel()

    counts.append({
        "Model": name,
        "Correct positives": tp,
        "False alarms": fp,
        "Missed positives": fn,
        "Correct negatives": tn
    })

display(pd.DataFrame(counts).set_index("Model"))

,Correct positives,False alarms,Missed positives,Correct negatives
Model,,,,
Logistic regression,6,46,12,289
CNN only,0,0,18,335
Combined model,0,0,18,335


In [ ]:
import pickle
from google.colab import files

checkpoint = {
    "baseline_model": baseline_model,
    "feature_scaler": feature_scaler,
    "feature_columns": feature_columns,
    "cnn_weights": cnn_model.state_dict(),
    "combined_weights": combined_model.state_dict(),
    "thresholds": model_thresholds,
    "training_guides": sorted(train_guides),
    "validation_guides": sorted(validation_guides),
    "test_guides": sorted(test_guides),
    "cnn_best_epoch": best_epoch,
    "combined_best_epoch": combined_best_epoch,
    "test_results": test_results
}

with open("EditGuard_first_run.pkl", "wb") as file:
    pickle.dump(checkpoint, file)

files.download("EditGuard_first_run.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
verified_data.to_csv(
    "EditGuard_prepared_data.csv",
    index=False
)

files.download("EditGuard_prepared_data.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>